# Nexus Runtime Notebook — State, Fold, Memory, Witness, Render

This notebook is the **first runnable substrate** for a Nexus-style system.

It does **not** try to solve AI all at once. It implements the first executable inversion:

\[
\text{state} \to \text{fold} \to \text{memory} \to \text{render}
\]

The notebook builds a minimal runtime with these components:

- **State**: the live system configuration
- **Fold operator**: the transition law
- **Memory store**: persistent state and recalled context
- **Witness ledger**: what actually happened
- **Trust engine**: a first accept/revise score
- **Renderer**: language output as a late-stage witness
- **Corpus ingest**: local text/markdown recall for memory recovery

The goal is simple:

1. persist a state
2. ingest a corpus
3. retrieve memory
4. process an input through the fold
5. score trust
6. write a witness log
7. render the result

From there, this notebook becomes the seed for a larger runtime, a service layer, and later model training.


## Notebook principles

This notebook follows a few rules:

- **Runtime first, model second**  
  The notebook works without a language model. A local model can be attached later.

- **Standard library first**  
  Core functionality uses Python's standard library so it is easy to run anywhere.

- **Persistence over vibes**  
  Every meaningful change gets written to SQLite.

- **Witness over guess**  
  The ledger records what actually happened so later training can distinguish recall from invention.

- **Portable design**  
  No hard-coded absolute paths. Everything is relative to the notebook root unless you override it.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sqlite3
import textwrap
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
from collections import Counter, defaultdict
from urllib import request, error

ROOT = Path.cwd()
DATA_DIR = ROOT / "nexus_runtime_data"
CORPUS_DIR = DATA_DIR / "corpus"
DB_PATH = DATA_DIR / "nexus.db"

DATA_DIR.mkdir(parents=True, exist_ok=True)
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("CORPUS_DIR:", CORPUS_DIR)
print("DB_PATH:", DB_PATH)


## 1. Core data objects

These are the primitive containers of the runtime:

- `NexusState`: the live fold-state
- `WitnessEvent`: an immutable event record
- `MemoryHit`: a recovered memory fragment


In [ ]:
@dataclass
class NexusState:
    state_id: str
    phase: str
    summary: str
    variables: Dict[str, Any] = field(default_factory=dict)
    active_branches: List[str] = field(default_factory=list)
    trust: float = 0.0
    residue: float = 0.0
    step_index: int = 0
    updated_at: str = field(default_factory=utc_now)

    def clone(self) -> "NexusState":
        return NexusState(
            state_id=self.state_id,
            phase=self.phase,
            summary=self.summary,
            variables=json.loads(json.dumps(self.variables)),
            active_branches=list(self.active_branches),
            trust=self.trust,
            residue=self.residue,
            step_index=self.step_index,
            updated_at=utc_now(),
        )


@dataclass
class WitnessEvent:
    state_id: str
    event_type: str
    payload: Dict[str, Any]
    created_at: str = field(default_factory=utc_now)


@dataclass
class MemoryHit:
    source_id: str
    source_type: str
    score: float
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)


## 2. SQLite-backed persistence

This store persists:

- states
- witness events
- corpus documents
- simple token statistics for retrieval


In [ ]:
class MemoryStore:
    def __init__(self, db_path: Path):
        self.db_path = Path(db_path)
        self.conn = sqlite3.connect(self.db_path)
        self.conn.row_factory = sqlite3.Row
        self._init_db()

    def _init_db(self) -> None:
        self.conn.executescript(
            '''
            CREATE TABLE IF NOT EXISTS states (
                state_id TEXT PRIMARY KEY,
                phase TEXT NOT NULL,
                summary TEXT NOT NULL,
                variables TEXT NOT NULL,
                active_branches TEXT NOT NULL,
                trust REAL NOT NULL,
                residue REAL NOT NULL,
                step_index INTEGER NOT NULL,
                updated_at TEXT NOT NULL
            );

            CREATE TABLE IF NOT EXISTS witness_log (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                state_id TEXT NOT NULL,
                event_type TEXT NOT NULL,
                payload TEXT NOT NULL,
                created_at TEXT NOT NULL
            );

            CREATE TABLE IF NOT EXISTS corpus_docs (
                doc_id TEXT PRIMARY KEY,
                path TEXT NOT NULL,
                source_type TEXT NOT NULL,
                title TEXT NOT NULL,
                text TEXT NOT NULL,
                metadata TEXT NOT NULL,
                indexed_at TEXT NOT NULL
            );

            CREATE TABLE IF NOT EXISTS corpus_terms (
                doc_id TEXT NOT NULL,
                term TEXT NOT NULL,
                tf REAL NOT NULL,
                PRIMARY KEY (doc_id, term)
            );
            '''
        )
        self.conn.commit()

    # ---------------------------
    # state operations
    # ---------------------------
    def save_state(self, state: NexusState) -> None:
        self.conn.execute(
            '''
            INSERT OR REPLACE INTO states
            (state_id, phase, summary, variables, active_branches, trust, residue, step_index, updated_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''',
            (
                state.state_id,
                state.phase,
                state.summary,
                json.dumps(state.variables),
                json.dumps(state.active_branches),
                state.trust,
                state.residue,
                state.step_index,
                state.updated_at,
            ),
        )
        self.conn.commit()

    def load_state(self, state_id: str) -> Optional[NexusState]:
        row = self.conn.execute(
            'SELECT * FROM states WHERE state_id = ?',
            (state_id,)
        ).fetchone()
        if row is None:
            return None
        return NexusState(
            state_id=row["state_id"],
            phase=row["phase"],
            summary=row["summary"],
            variables=json.loads(row["variables"]),
            active_branches=json.loads(row["active_branches"]),
            trust=row["trust"],
            residue=row["residue"],
            step_index=row["step_index"],
            updated_at=row["updated_at"],
        )

    def list_states(self) -> List[NexusState]:
        rows = self.conn.execute(
            'SELECT * FROM states ORDER BY updated_at DESC'
        ).fetchall()
        return [
            NexusState(
                state_id=row["state_id"],
                phase=row["phase"],
                summary=row["summary"],
                variables=json.loads(row["variables"]),
                active_branches=json.loads(row["active_branches"]),
                trust=row["trust"],
                residue=row["residue"],
                step_index=row["step_index"],
                updated_at=row["updated_at"],
            )
            for row in rows
        ]

    # ---------------------------
    # witness operations
    # ---------------------------
    def record_event(self, event: WitnessEvent) -> None:
        self.conn.execute(
            '''
            INSERT INTO witness_log (state_id, event_type, payload, created_at)
            VALUES (?, ?, ?, ?)
            ''',
            (
                event.state_id,
                event.event_type,
                json.dumps(event.payload),
                event.created_at,
            ),
        )
        self.conn.commit()

    def get_events(self, state_id: Optional[str] = None, limit: int = 20) -> List[dict]:
        if state_id:
            rows = self.conn.execute(
                '''
                SELECT * FROM witness_log
                WHERE state_id = ?
                ORDER BY id DESC
                LIMIT ?
                ''',
                (state_id, limit),
            ).fetchall()
        else:
            rows = self.conn.execute(
                '''
                SELECT * FROM witness_log
                ORDER BY id DESC
                LIMIT ?
                ''',
                (limit,),
            ).fetchall()

        return [
            {
                "id": row["id"],
                "state_id": row["state_id"],
                "event_type": row["event_type"],
                "payload": json.loads(row["payload"]),
                "created_at": row["created_at"],
            }
            for row in rows
        ]

    # ---------------------------
    # corpus operations
    # ---------------------------
    def upsert_corpus_doc(
        self,
        doc_id: str,
        path: str,
        source_type: str,
        title: str,
        text: str,
        metadata: Dict[str, Any],
        term_weights: Dict[str, float],
    ) -> None:
        self.conn.execute(
            '''
            INSERT OR REPLACE INTO corpus_docs
            (doc_id, path, source_type, title, text, metadata, indexed_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            ''',
            (
                doc_id,
                path,
                source_type,
                title,
                text,
                json.dumps(metadata),
                utc_now(),
            ),
        )
        self.conn.execute('DELETE FROM corpus_terms WHERE doc_id = ?', (doc_id,))
        self.conn.executemany(
            '''
            INSERT INTO corpus_terms (doc_id, term, tf)
            VALUES (?, ?, ?)
            ''',
            [(doc_id, term, weight) for term, weight in term_weights.items()],
        )
        self.conn.commit()

    def fetch_doc(self, doc_id: str) -> Optional[dict]:
        row = self.conn.execute(
            'SELECT * FROM corpus_docs WHERE doc_id = ?',
            (doc_id,),
        ).fetchone()
        if row is None:
            return None
        return {
            "doc_id": row["doc_id"],
            "path": row["path"],
            "source_type": row["source_type"],
            "title": row["title"],
            "text": row["text"],
            "metadata": json.loads(row["metadata"]),
            "indexed_at": row["indexed_at"],
        }

    def list_docs(self, limit: int = 20) -> List[dict]:
        rows = self.conn.execute(
            'SELECT * FROM corpus_docs ORDER BY indexed_at DESC LIMIT ?',
            (limit,),
        ).fetchall()
        return [
            {
                "doc_id": row["doc_id"],
                "path": row["path"],
                "source_type": row["source_type"],
                "title": row["title"],
                "text": row["text"],
                "metadata": json.loads(row["metadata"]),
                "indexed_at": row["indexed_at"],
            }
            for row in rows
        ]


memory_store = MemoryStore(DB_PATH)
print("Database initialized.")


## 3. Corpus ingest and recall

This is a deliberately simple retrieval layer:

- tokenize text
- compute normalized term frequencies
- retrieve by query overlap

That is enough for a first runtime. You can replace it later with BM25, embeddings, hybrid search, or graph retrieval.


In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z0-9_\-]+")

def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: str) -> List[str]:
    return [t.lower() for t in TOKEN_RE.findall(text)]

def term_weights(text: str) -> Dict[str, float]:
    tokens = tokenize(text)
    if not tokens:
        return {}
    counts = Counter(tokens)
    total = sum(counts.values())
    return {term: count / total for term, count in counts.items()}

def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 200) -> List[str]:
    text = text.strip()
    if len(text) <= chunk_size:
        return [text] if text else []
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

class CorpusIndexer:
    def __init__(self, store: MemoryStore):
        self.store = store

    def ingest_paths(self, paths: Iterable[Path], source_type: str = "file") -> int:
        count = 0
        for path in paths:
            path = Path(path)
            if not path.exists() or not path.is_file():
                continue

            text = path.read_text(encoding="utf-8", errors="ignore")
            text = normalize_text(text)
            if not text:
                continue

            chunks = chunk_text(text)
            for idx, chunk in enumerate(chunks):
                doc_id = f"{path.stem}:{idx}"
                title = f"{path.name} [chunk {idx}]"
                metadata = {"file_name": path.name, "chunk_index": idx}
                self.store.upsert_corpus_doc(
                    doc_id=doc_id,
                    path=str(path),
                    source_type=source_type,
                    title=title,
                    text=chunk,
                    metadata=metadata,
                    term_weights=term_weights(chunk),
                )
                count += 1
        return count

class CorpusRetriever:
    def __init__(self, store: MemoryStore):
        self.store = store

    def search(self, query: str, limit: int = 5) -> List[MemoryHit]:
        q_weights = term_weights(query)
        if not q_weights:
            return []

        scores = defaultdict(float)
        cur = self.store.conn.execute(
            'SELECT doc_id, term, tf FROM corpus_terms'
        )
        for row in cur:
            term = row["term"]
            if term in q_weights:
                scores[row["doc_id"]] += q_weights[term] * row["tf"]

        hits = []
        for doc_id, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:limit]:
            doc = self.store.fetch_doc(doc_id)
            if not doc:
                continue
            hits.append(
                MemoryHit(
                    source_id=doc["doc_id"],
                    source_type=doc["source_type"],
                    score=score,
                    text=doc["text"],
                    metadata={
                        "title": doc["title"],
                        "path": doc["path"],
                        **doc["metadata"],
                    },
                )
            )
        return hits


indexer = CorpusIndexer(memory_store)
retriever = CorpusRetriever(memory_store)
print("Indexer and retriever ready.")


## 4. Create a starter corpus

This cell writes a few seed documents into the local corpus folder so the notebook is runnable out of the box.

You can replace or expand these with your own markdown, text, notebooks, logs, or exports later.


In [ ]:
starter_docs = {
    "ontology.md": '''
    Nexus runtime seed:
    State comes before render.
    Memory is retained curvature from reusable change.
    Witness is what actually happened.
    Trust is the score that separates stable folds from residue.
    Render is a late-stage projection, not the ground.
    ''',
    "runtime.md": '''
    Canonical runtime equation:
    S_{t+1} = F(S_t, U_t, M_t)

    First build:
    - persistent state
    - witness log
    - corpus retrieval
    - trust engine
    - renderer
    ''',
    "memory.md": '''
    Lost memory recovery is not database lookup alone.
    It is reconstruction from prior curvature, witness traces, and recalled fragments.
    The runtime should distinguish:
    - witnessed memory
    - inferred memory
    - hallucinated memory
    ''',
}

for name, text in starter_docs.items():
    (CORPUS_DIR / name).write_text(textwrap.dedent(text).strip() + "\n", encoding="utf-8")

added = indexer.ingest_paths(sorted(CORPUS_DIR.glob("*.md")))
print(f"Indexed {added} corpus chunks.")
print("Recent docs:")
for doc in memory_store.list_docs(limit=10):
    print("-", doc["title"])


In [ ]:
hits = retriever.search("memory recovery from witness and retained curvature", limit=5)
for hit in hits:
    print("=" * 80)
    print(hit.metadata.get("title"))
    print("score:", round(hit.score, 6))
    print(hit.text[:400])


## 5. Trust engine

The first trust engine is intentionally simple.

It does not claim universal truth. It just penalizes obvious instability:
- empty summary
- no memory support
- high residue
- branch explosion

You will replace this with stronger scoring later.


In [ ]:
class TrustEngine:
    def score(
        self,
        proposed_summary: str,
        memory_hits: List[MemoryHit],
        residue: float,
        active_branches: List[str],
    ) -> Tuple[float, Dict[str, Any]]:
        score = 1.0
        reasons = []

        if not proposed_summary or len(proposed_summary.strip()) < 10:
            score -= 0.35
            reasons.append("summary_too_thin")

        if not memory_hits:
            score -= 0.20
            reasons.append("no_recalled_memory")

        if residue > 0.50:
            score -= 0.25
            reasons.append("high_residue")

        if len(active_branches) > 8:
            score -= 0.10
            reasons.append("too_many_branches")

        score = max(0.0, min(1.0, score))
        diagnostics = {
            "reasons": reasons,
            "memory_hits": len(memory_hits),
            "residue": residue,
            "branch_count": len(active_branches),
        }
        return score, diagnostics


trust_engine = TrustEngine()
print("Trust engine ready.")


## 6. Optional local model adapter

This adapter tries to call a local Ollama instance.

It is **optional**. If Ollama is not running, the runtime still works and falls back to a deterministic renderer.

Default endpoint:
- `http://localhost:11434/api/generate`

Default model:
- `qwen3:8b`


In [ ]:
class OllamaRenderer:
    def __init__(self, model: str = "qwen3:8b", endpoint: str = "http://localhost:11434/api/generate"):
        self.model = model
        self.endpoint = endpoint

    def generate(self, prompt: str, timeout: int = 60) -> Optional[str]:
        payload = json.dumps({
            "model": self.model,
            "prompt": prompt,
            "stream": False,
        }).encode("utf-8")

        req = request.Request(
            self.endpoint,
            data=payload,
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        try:
            with request.urlopen(req, timeout=timeout) as resp:
                data = json.loads(resp.read().decode("utf-8"))
                return data.get("response", "").strip()
        except Exception as exc:
            print(f"[Ollama unavailable] {exc}")
            return None


ollama_renderer = OllamaRenderer()


## 7. Renderer

If no local model is available, the renderer produces a structured textual report from the current state.

If Ollama is reachable, it can generate a richer natural-language rendering on top of the same state.


In [ ]:
class Renderer:
    def __init__(self, llm: Optional[OllamaRenderer] = None):
        self.llm = llm

    def deterministic_render(self, state: NexusState, memory_hits: List[MemoryHit], diagnostics: Dict[str, Any]) -> str:
        lines = [
            f"state_id: {state.state_id}",
            f"phase: {state.phase}",
            f"step_index: {state.step_index}",
            f"trust: {state.trust:.3f}",
            f"residue: {state.residue:.3f}",
            f"summary: {state.summary}",
            f"branches: {state.active_branches}",
            f"diagnostics: {diagnostics}",
        ]
        if memory_hits:
            lines.append("recalled_memory:")
            for hit in memory_hits:
                title = hit.metadata.get("title", hit.source_id)
                excerpt = hit.text[:180].replace("\n", " ")
                lines.append(f"  - {title} | score={hit.score:.4f} | {excerpt}")
        return "\n".join(lines)

    def llm_render(self, state: NexusState, memory_hits: List[MemoryHit], diagnostics: Dict[str, Any]) -> Optional[str]:
        if self.llm is None:
            return None

        memory_block = "\n".join(
            f"- {hit.metadata.get('title', hit.source_id)} :: {hit.text[:250]}"
            for hit in memory_hits
        ) or "(none)"

        prompt = f'''
        You are rendering a runtime state report.
        Do not invent facts.
        Use only the supplied state and recalled memory.

        STATE
        {json.dumps(asdict(state), indent=2)}

        DIAGNOSTICS
        {json.dumps(diagnostics, indent=2)}

        RECALLED MEMORY
        {memory_block}

        Write a concise state report with:
        1. current fold
        2. recalled memory relevance
        3. trust and residue
        4. next safe action
        '''
        return self.llm.generate(textwrap.dedent(prompt).strip())

    def render(self, state: NexusState, memory_hits: List[MemoryHit], diagnostics: Dict[str, Any], use_llm: bool = False) -> str:
        if use_llm:
            result = self.llm_render(state, memory_hits, diagnostics)
            if result:
                return result
        return self.deterministic_render(state, memory_hits, diagnostics)


renderer = Renderer(llm=ollama_renderer)
print("Renderer ready.")


## 8. Fold operator

This is the first executable transition law.

For now, the fold does four things:

1. recover memory from the corpus
2. update summary and variables
3. estimate residue
4. increment the live state

This is minimal by design. The point is to make the ontology runnable.


In [ ]:
def estimate_residue(user_input: str, memory_hits: List[MemoryHit]) -> float:
    if not user_input.strip():
        return 1.0
    if not memory_hits:
        return 0.65
    best = max(hit.score for hit in memory_hits)
    residue = 1.0 - min(1.0, best * 10.0)
    return max(0.0, min(1.0, residue))

class FoldOperator:
    def __init__(self, retriever: CorpusRetriever):
        self.retriever = retriever

    def step(self, state: NexusState, user_input: str) -> Tuple[NexusState, List[MemoryHit]]:
        next_state = state.clone()
        next_state.step_index += 1
        next_state.updated_at = utc_now()

        memory_hits = self.retriever.search(user_input, limit=5)
        next_state.residue = estimate_residue(user_input, memory_hits)

        next_state.variables["last_input"] = user_input
        next_state.variables["last_memory_ids"] = [hit.source_id for hit in memory_hits]
        next_state.variables["last_memory_titles"] = [hit.metadata.get("title", hit.source_id) for hit in memory_hits]

        if memory_hits:
            top_titles = ", ".join(hit.metadata.get("title", hit.source_id) for hit in memory_hits[:3])
            next_state.summary = f"Input coupled to recalled memory: {top_titles}"
            next_state.phase = "memory_coupled"
        else:
            next_state.summary = "Input processed without memory support"
            next_state.phase = "weak_coupling"

        if next_state.residue > 0.7:
            next_state.active_branches = list(set(next_state.active_branches + ["quarantine_review"]))
        elif "quarantine_review" in next_state.active_branches:
            next_state.active_branches = [b for b in next_state.active_branches if b != "quarantine_review"]

        return next_state, memory_hits


fold_operator = FoldOperator(retriever)
print("Fold operator ready.")


## 9. Witness ledger wrapper

The ledger records the full transition so later training can learn from:
- input
- recalled memory
- trust score
- diagnostics
- rendered output

That is the seed of a real post-training corpus.


In [ ]:
class WitnessLedger:
    def __init__(self, store: MemoryStore):
        self.store = store

    def record_transition(
        self,
        state_before: NexusState,
        state_after: NexusState,
        user_input: str,
        memory_hits: List[MemoryHit],
        diagnostics: Dict[str, Any],
        rendered_output: str,
    ) -> None:
        event = WitnessEvent(
            state_id=state_after.state_id,
            event_type="transition",
            payload={
                "before": asdict(state_before),
                "after": asdict(state_after),
                "input": user_input,
                "memory_hits": [
                    {
                        "source_id": hit.source_id,
                        "source_type": hit.source_type,
                        "score": hit.score,
                        "metadata": hit.metadata,
                        "text_preview": hit.text[:250],
                    }
                    for hit in memory_hits
                ],
                "diagnostics": diagnostics,
                "rendered_output": rendered_output,
            },
        )
        self.store.record_event(event)


ledger = WitnessLedger(memory_store)
print("Witness ledger ready.")


## 10. Runtime orchestrator

This class ties everything together.

It is the first real **Nexus runtime loop**:
- load state
- fold
- trust
- render
- log witness
- persist state


In [ ]:
class NexusRuntime:
    def __init__(
        self,
        store: MemoryStore,
        fold_operator: FoldOperator,
        trust_engine: TrustEngine,
        renderer: Renderer,
        ledger: WitnessLedger,
        state_id: str = "root",
    ):
        self.store = store
        self.fold_operator = fold_operator
        self.trust_engine = trust_engine
        self.renderer = renderer
        self.ledger = ledger
        self.state_id = state_id

        state = self.store.load_state(self.state_id)
        if state is None:
            state = NexusState(
                state_id=self.state_id,
                phase="genesis",
                summary="Initial Nexus runtime state",
            )
            self.store.save_state(state)
        self.state = state

    def step(self, user_input: str, use_llm: bool = False) -> Dict[str, Any]:
        before = self.state.clone()
        after, memory_hits = self.fold_operator.step(before, user_input)

        trust, diagnostics = self.trust_engine.score(
            proposed_summary=after.summary,
            memory_hits=memory_hits,
            residue=after.residue,
            active_branches=after.active_branches,
        )
        after.trust = trust

        rendered = self.renderer.render(after, memory_hits, diagnostics, use_llm=use_llm)

        self.store.save_state(after)
        self.ledger.record_transition(
            state_before=before,
            state_after=after,
            user_input=user_input,
            memory_hits=memory_hits,
            diagnostics=diagnostics,
            rendered_output=rendered,
        )

        self.state = after
        return {
            "state": after,
            "memory_hits": memory_hits,
            "diagnostics": diagnostics,
            "rendered": rendered,
        }

    def reload(self) -> NexusState:
        self.state = self.store.load_state(self.state_id)
        return self.state


runtime = NexusRuntime(
    store=memory_store,
    fold_operator=fold_operator,
    trust_engine=trust_engine,
    renderer=renderer,
    ledger=ledger,
)
print("Runtime state loaded:")
print(runtime.state)


## 11. Run a few steps

This demonstrates the loop without requiring a local LLM.


In [ ]:
demo_inputs = [
    "We need to recover lost memory from witness traces.",
    "State comes before language output.",
    "What is the next safe action for the runtime?",
]

results = []
for q in demo_inputs:
    result = runtime.step(q, use_llm=False)
    results.append(result)
    print("=" * 100)
    print("INPUT:", q)
    print(result["rendered"])


## 12. Inspect saved state and witness events


In [ ]:
print("Current state:")
print(runtime.reload())

print("\nRecent witness events:")
events = memory_store.get_events(state_id="root", limit=5)
for evt in events:
    print("-" * 100)
    print("event_id:", evt["id"])
    print("created_at:", evt["created_at"])
    print("event_type:", evt["event_type"])
    print("input:", evt["payload"]["input"])
    print("after.summary:", evt["payload"]["after"]["summary"])
    print("diagnostics:", evt["payload"]["diagnostics"])


## 13. Optional: try local Ollama rendering

If you have Ollama running locally and a model available, turn `use_llm=True`.

Example:
- install Ollama
- pull a model such as `qwen3:8b`
- start the Ollama service
- rerun the next cell


In [ ]:
# Uncomment to try LLM-backed rendering.
# result = runtime.step("Summarize the current state and next safe action.", use_llm=True)
# print(result["rendered"])


## 14. Minimal memory-recovery benchmark

A runtime rewrite needs a measurable test.

This benchmark asks a simple question:

> Does retrieval improve state formation for memory-bearing prompts?

Right now it measures only whether memory was recalled and whether trust stayed above a threshold.
That is crude, but it gives you a concrete starting metric.


In [ ]:
def run_memory_benchmark(runtime: NexusRuntime, prompts: List[str], trust_threshold: float = 0.60) -> List[dict]:
    rows = []
    for prompt in prompts:
        result = runtime.step(prompt, use_llm=False)
        rows.append({
            "prompt": prompt,
            "memory_hits": len(result["memory_hits"]),
            "trust": result["state"].trust,
            "residue": result["state"].residue,
            "passes_threshold": result["state"].trust >= trust_threshold,
            "phase": result["state"].phase,
        })
    return rows

benchmark_prompts = [
    "recover memory from witness traces",
    "render state after recall",
    "invent something unsupported",
    "retained curvature and memory recovery",
]

benchmark_rows = run_memory_benchmark(runtime, benchmark_prompts)
benchmark_rows


In [ ]:
for row in benchmark_rows:
    print(row)


## 15. Where this goes next

This notebook is **Nexus-0**: the first executable ontological container.

The next stages are:

### Nexus-1
Replace simple keyword retrieval with:
- BM25
- embeddings
- hybrid retrieval
- graph memory

### Nexus-2
Replace the simple fold with:
- branch control
- explicit quarantine
- multi-step correction
- state reconciliation

### Nexus-3
Attach a local model cleanly:
- Ollama
- llama.cpp server
- an API-compatible local endpoint

### Nexus-4
Train on witness traces:
- input
- recalled memory
- transition
- trust outcome
- rendered output

That training set is the first real substrate for a future rewritten AI.


## 16. Action list

What to do immediately after this notebook runs:

1. **Run the notebook once end-to-end**
2. **Replace the starter corpus with your own markdown/text files**
3. **Verify that state persists across reruns**
4. **Inspect the witness log**
5. **Attach a local model only after the runtime works without one**
6. **Export witness traces as JSONL for later training**
7. **Only then move heavier experiments to pods**

That is the correct order.

You are not coding the final mind yet.

You are coding the first **runtime that a different kind of mind could live inside**.
